## Check whether price_band is target leakage

In [1]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(
    "/Users/souravkumar/Downloads/"
    "AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System"
)

TRAIN_FILE = (
    PROJECT_ROOT
    / "data"
    / "amazon_multimodal"
    / "model_input"
    / "train.parquet"
)

train_df = pd.read_parquet(TRAIN_FILE)

print("Shape:", train_df.shape)

print("\nColumns:")
for col in train_df.columns:
    print(col)

print("\nPrice-band examples:")
display(
    train_df[
        ["price", "price_band"]
    ].head(30)
)

print("\nPrice statistics by price_band:")
display(
    train_df.groupby("price_band")["price"]
    .agg([
        "count",
        "min",
        "max",
        "mean",
        "median",
    ])
    .sort_values("min")
)

Shape: (13984, 26)

Columns:
asin
title
category_name
price
listPrice
stars
reviews
isBestSeller
boughtInLastMonth
product_text
price_band
cluster_id
imgUrl
productURL
absolute_path
download_success
download_status
width
height
file_size_bytes
image_exists
model_text
image_file_exists
original_split_group
split_group
dataset_split

Price-band examples:


,price,price_band
0,7.99,very_low
1,45.99,premium
2,13.99,low
3,86.99,premium
4,13.49,low
5,20.99,medium
6,39.99,high
7,125.00,premium
8,9.27,very_low
9,7.99,very_low



Price statistics by price_band:


,count,min,max,mean,median
price_band,,,,,
very_low,2388,0.75,10.71,8.573564,8.99
low,2600,10.75,16.48,13.776588,13.99
medium,2741,16.49,24.99,20.219847,19.99
high,2813,25.00,43.99,32.625542,30.00
premium,2439,44.00,144.97,66.409077,54.99
luxury,1003,144.99,8499.99,286.599761,188.37


In [3]:
import joblib
import pandas as pd
from pathlib import Path


# ============================================================
# PATHS
# ============================================================

CURRENT_DIRECTORY = Path.cwd()

PROJECT_ROOT = (
    CURRENT_DIRECTORY.parent
    if CURRENT_DIRECTORY.name.lower() == "notebooks"
    else CURRENT_DIRECTORY
)

MODEL_ROOT = (
    PROJECT_ROOT
    / "models"
    / "price_prediction"
)

PREPROCESSOR_FILE = (
    MODEL_ROOT
    / "structured_preprocessor.joblib"
)

TRAIN_FILE = (
    PROJECT_ROOT
    / "data"
    / "amazon_multimodal"
    / "model_input"
    / "train.parquet"
)


# ============================================================
# LOAD
# ============================================================

if not PREPROCESSOR_FILE.exists():
    raise FileNotFoundError(
        f"Preprocessor not found: {PREPROCESSOR_FILE}"
    )

if not TRAIN_FILE.exists():
    raise FileNotFoundError(
        f"Train file not found: {TRAIN_FILE}"
    )

preprocessor = joblib.load(
    PREPROCESSOR_FILE
)

train_df = pd.read_parquet(
    TRAIN_FILE
)


# ============================================================
# SHOW ORIGINAL DATA COLUMNS
# ============================================================

print("=" * 80)
print("TRAIN DATA COLUMNS")
print("=" * 80)

for column in train_df.columns:
    print(column)


# ============================================================
# SHOW PREPROCESSOR INPUT FEATURES
# ============================================================

print()
print("=" * 80)
print("FEATURES USED BY STRUCTURED PREPROCESSOR")
print("=" * 80)

for transformer_name, transformer, columns in preprocessor.transformers_:

    if transformer_name == "remainder":
        continue

    print()
    print(f"{transformer_name.upper()} FEATURES:")

    for column in columns:
        print("  -", column)


# ============================================================
# CHECK SUSPICIOUS COLUMNS
# ============================================================

used_columns = []

for transformer_name, transformer, columns in preprocessor.transformers_:

    if transformer_name == "remainder":
        continue

    used_columns.extend(
        list(columns)
    )


suspicious_columns = [
    "price",
    "price_band",
    "listPrice",
    "productURL",
    "imgUrl",
    "absolute_path",
    "dataset_split",
    "split_group",
    "original_split_group",
]


print()
print("=" * 80)
print("LEAKAGE / SUSPICIOUS FEATURE CHECK")
print("=" * 80)

for column in suspicious_columns:

    if column in used_columns:
        print(
            f"⚠️ USED IN MODEL: {column}"
        )

    else:
        print(
            f"✅ NOT USED: {column}"
        )


# ============================================================
# FEATURE COUNT
# ============================================================

try:

    feature_names = (
        preprocessor
        .get_feature_names_out()
    )

    print()
    print("=" * 80)
    print("FINAL ENCODED FEATURES")
    print("=" * 80)

    print(
        "Total encoded features:",
        len(feature_names)
    )

    print()
    print("First 50 features:")

    for feature in feature_names[:50]:
        print(feature)

except Exception as error:

    print(
        "Could not extract encoded feature names:",
        error
    )

TRAIN DATA COLUMNS
asin
title
category_name
price
listPrice
stars
reviews
isBestSeller
boughtInLastMonth
product_text
price_band
cluster_id
imgUrl
productURL
absolute_path
download_success
download_status
width
height
file_size_bytes
image_exists
model_text
image_file_exists
original_split_group
split_group
dataset_split

FEATURES USED BY STRUCTURED PREPROCESSOR

NUMERIC FEATURES:
  - stars
  - reviews_log1p
  - bought_log1p
  - isBestSeller
  - cluster_id

CATEGORICAL FEATURES:
  - category_name
  - price_band

LEAKAGE / SUSPICIOUS FEATURE CHECK
✅ NOT USED: price
⚠️ USED IN MODEL: price_band
✅ NOT USED: listPrice
✅ NOT USED: productURL
✅ NOT USED: imgUrl
✅ NOT USED: absolute_path
✅ NOT USED: dataset_split
✅ NOT USED: split_group
✅ NOT USED: original_split_group

FINAL ENCODED FEATURES
Total encoded features: 251

First 50 features:
numeric__stars
numeric__reviews_log1p
numeric__bought_log1p
numeric__isBestSeller
numeric__cluster_id
categorical__category_name_Abrasive & Finishing Produ

## structured features without price_band

In [4]:
from __future__ import annotations

import gc
from pathlib import Path

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    r2_score,
)
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)


# ============================================================
# CONFIGURATION
# ============================================================

CURRENT_DIRECTORY = Path.cwd()

PROJECT_ROOT = (
    CURRENT_DIRECTORY.parent
    if CURRENT_DIRECTORY.name.lower() == "notebooks"
    else CURRENT_DIRECTORY
)

MODEL_INPUT_ROOT = (
    PROJECT_ROOT
    / "data"
    / "amazon_multimodal"
    / "model_input"
)

MODEL_OUTPUT_ROOT = (
    PROJECT_ROOT
    / "models"
    / "price_prediction"
    / "leakage_free"
)

REPORT_ROOT = (
    PROJECT_ROOT
    / "data"
    / "reports"
    / "price_prediction"
    / "leakage_free"
)

MODEL_OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

REPORT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

RANDOM_STATE = 42


# ============================================================
# LOAD DATA
# ============================================================

train_df = pd.read_parquet(
    MODEL_INPUT_ROOT / "train.parquet"
)

validation_df = pd.read_parquet(
    MODEL_INPUT_ROOT / "validation.parquet"
)

test_df = pd.read_parquet(
    MODEL_INPUT_ROOT / "test.parquet"
)

print("=" * 80)
print("DATASET")
print("=" * 80)

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)


# ============================================================
# FEATURE ENGINEERING
# ============================================================

def create_features(df):

    df = df.copy()

    df["reviews_log1p"] = np.log1p(
        pd.to_numeric(
            df["reviews"],
            errors="coerce",
        ).fillna(0)
    )

    df["bought_log1p"] = np.log1p(
        pd.to_numeric(
            df["boughtInLastMonth"],
            errors="coerce",
        ).fillna(0)
    )

    df["stars"] = pd.to_numeric(
        df["stars"],
        errors="coerce",
    ).fillna(0)

    df["isBestSeller"] = (
        df["isBestSeller"]
        .fillna(False)
        .astype(int)
    )

    df["cluster_id"] = pd.to_numeric(
        df["cluster_id"],
        errors="coerce",
    ).fillna(-1)

    df["category_name"] = (
        df["category_name"]
        .fillna("Unknown")
        .astype(str)
    )

    return df


train_df = create_features(train_df)
validation_df = create_features(validation_df)
test_df = create_features(test_df)


# ============================================================
# CLEAN FEATURES
# ============================================================

NUMERIC_FEATURES = [
    "stars",
    "reviews_log1p",
    "bought_log1p",
    "isBestSeller",
    "cluster_id",
]

CATEGORICAL_FEATURES = [
    "category_name",
]

# IMPORTANT:
#
# price       -> TARGET
# price_band  -> ANALYSIS ONLY
# listPrice   -> NOT USED YET


print()
print("=" * 80)
print("LEAKAGE-FREE FEATURE CONFIGURATION")
print("=" * 80)

print("\nNumeric:")
for feature in NUMERIC_FEATURES:
    print("  ", feature)

print("\nCategorical:")
for feature in CATEGORICAL_FEATURES:
    print("  ", feature)

print("\nExcluded:")
print("   price       -> target")
print("   price_band  -> target leakage")
print("   listPrice   -> excluded from clean baseline")


# ============================================================
# PREPROCESSOR
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            NUMERIC_FEATURES,
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
            CATEGORICAL_FEATURES,
        ),
    ]
)


X_train = preprocessor.fit_transform(
    train_df
).astype(np.float32)

X_validation = preprocessor.transform(
    validation_df
).astype(np.float32)

X_test = preprocessor.transform(
    test_df
).astype(np.float32)


y_train = (
    train_df["price"]
    .astype(np.float32)
    .to_numpy()
)

y_validation = (
    validation_df["price"]
    .astype(np.float32)
    .to_numpy()
)

y_test = (
    test_df["price"]
    .astype(np.float32)
    .to_numpy()
)


print()
print("=" * 80)
print("FINAL FEATURE SHAPES")
print("=" * 80)

print("Train:", X_train.shape)
print("Validation:", X_validation.shape)
print("Test:", X_test.shape)


# ============================================================
# VERIFY LEAKAGE REMOVAL
# ============================================================

feature_names = (
    preprocessor
    .get_feature_names_out()
)

leaked_features = [
    feature
    for feature in feature_names
    if "price_band" in feature.lower()
]

if leaked_features:

    raise RuntimeError(
        "price_band is still present!"
    )

print()
print(
    "✅ price_band successfully removed "
    "from model features."
)

print(
    "Total clean features:",
    len(feature_names),
)


# ============================================================
# SAVE PREPROCESSOR
# ============================================================

joblib.dump(
    preprocessor,
    MODEL_OUTPUT_ROOT
    / "clean_structured_preprocessor.joblib",
)


# ============================================================
# METRICS
# ============================================================

def evaluate(
    model_name,
    dataset_name,
    y_true,
    predictions,
):

    mae = mean_absolute_error(
        y_true,
        predictions,
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            predictions,
        )
    )

    median_ae = median_absolute_error(
        y_true,
        predictions,
    )

    r2 = r2_score(
        y_true,
        predictions,
    )

    print()
    print(
        f"{model_name} - {dataset_name}"
    )

    print("-" * 60)

    print(f"MAE:       {mae:.4f}")
    print(f"RMSE:      {rmse:.4f}")
    print(f"Median AE: {median_ae:.4f}")
    print(f"R²:        {r2:.4f}")

    return {
        "model": model_name,
        "dataset": dataset_name,
        "mae": mae,
        "rmse": rmse,
        "median_absolute_error": median_ae,
        "r2": r2,
    }


results = []
test_predictions = pd.DataFrame(
    {
        "asin": test_df["asin"].values,
        "actual_price": y_test,
        "price_band": test_df[
            "price_band"
        ].values,
        "category_name": test_df[
            "category_name"
        ].values,
        "cluster_id": test_df[
            "cluster_id"
        ].values,
    }
)


# ============================================================
# 1. RANDOM FOREST
# ============================================================

print()
print("=" * 80)
print("TRAINING RANDOM FOREST")
print("=" * 80)

rf_model = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=2,
    max_features="sqrt",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

rf_model.fit(
    X_train,
    y_train,
)

rf_validation_predictions = (
    rf_model.predict(
        X_validation
    )
)

rf_test_predictions = (
    rf_model.predict(
        X_test
    )
)

results.append(
    evaluate(
        "RandomForest",
        "validation",
        y_validation,
        rf_validation_predictions,
    )
)

results.append(
    evaluate(
        "RandomForest",
        "test",
        y_test,
        rf_test_predictions,
    )
)

test_predictions[
    "random_forest_prediction"
] = rf_test_predictions

joblib.dump(
    rf_model,
    MODEL_OUTPUT_ROOT
    / "random_forest.joblib",
)

del rf_model
gc.collect()


# ============================================================
# 2. XGBOOST
# ============================================================

print()
print("=" * 80)
print("TRAINING XGBOOST")
print("=" * 80)

xgb_model = xgb.XGBRegressor(
    n_estimators=3000,
    learning_rate=0.03,
    max_depth=7,

    subsample=0.8,
    colsample_bytree=0.8,

    reg_alpha=0.05,
    reg_lambda=1.0,

    objective="reg:squarederror",

    random_state=RANDOM_STATE,
    n_jobs=-1,

    early_stopping_rounds=100,
)

xgb_model.fit(
    X_train,
    y_train,

    eval_set=[
        (
            X_validation,
            y_validation,
        )
    ],

    verbose=False,
)

xgb_validation_predictions = (
    xgb_model.predict(
        X_validation
    )
)

xgb_test_predictions = (
    xgb_model.predict(
        X_test
    )
)

results.append(
    evaluate(
        "XGBoost",
        "validation",
        y_validation,
        xgb_validation_predictions,
    )
)

results.append(
    evaluate(
        "XGBoost",
        "test",
        y_test,
        xgb_test_predictions,
    )
)

test_predictions[
    "xgboost_prediction"
] = xgb_test_predictions

joblib.dump(
    xgb_model,
    MODEL_OUTPUT_ROOT
    / "xgboost.joblib",
)

del xgb_model
gc.collect()


# ============================================================
# 3. LIGHTGBM
# ============================================================

print()
print("=" * 80)
print("TRAINING LIGHTGBM")
print("=" * 80)

lgb_model = lgb.LGBMRegressor(
    n_estimators=3000,

    learning_rate=0.03,

    num_leaves=63,

    max_depth=-1,

    min_child_samples=20,

    subsample=0.8,

    colsample_bytree=0.8,

    reg_alpha=0.05,

    reg_lambda=1.0,

    random_state=RANDOM_STATE,

    verbosity=-1,
)

lgb_model.fit(
    X_train,
    y_train,

    eval_set=[
        (
            X_validation,
            y_validation,
        )
    ],

    callbacks=[
        lgb.early_stopping(
            100,
            verbose=False,
        )
    ],
)

lgb_validation_predictions = (
    lgb_model.predict(
        X_validation
    )
)

lgb_test_predictions = (
    lgb_model.predict(
        X_test
    )
)

results.append(
    evaluate(
        "LightGBM",
        "validation",
        y_validation,
        lgb_validation_predictions,
    )
)

results.append(
    evaluate(
        "LightGBM",
        "test",
        y_test,
        lgb_test_predictions,
    )
)

test_predictions[
    "lightgbm_prediction"
] = lgb_test_predictions


joblib.dump(
    lgb_model,
    MODEL_OUTPUT_ROOT
    / "lightgbm.joblib",
)


# ============================================================
# MODEL COMPARISON
# ============================================================

results_df = pd.DataFrame(
    results
)

validation_results = (
    results_df[
        results_df["dataset"]
        == "validation"
    ]
    .sort_values(
        "mae"
    )
    .reset_index(
        drop=True
    )
)


print()
print("=" * 80)
print("LEAKAGE-FREE MODEL RANKING")
print("=" * 80)

display(
    validation_results
)


# ============================================================
# SAVE RESULTS
# ============================================================

results_df.to_csv(
    REPORT_ROOT
    / "clean_model_comparison.csv",
    index=False,
)

test_predictions.to_csv(
    REPORT_ROOT
    / "clean_test_predictions.csv",
    index=False,
)


# ============================================================
# WINNER
# ============================================================

best = validation_results.iloc[0]

print()
print("=" * 80)
print("BEST LEAKAGE-FREE MODEL")
print("=" * 80)

print(
    "Model:",
    best["model"],
)

print(
    f"Validation MAE: "
    f"{best['mae']:.4f}"
)

print(
    f"Validation RMSE: "
    f"{best['rmse']:.4f}"
)

print(
    f"Validation Median AE: "
    f"{best['median_absolute_error']:.4f}"
)

print(
    f"Validation R²: "
    f"{best['r2']:.4f}"
)


print()
print("=" * 80)
print("COMPLETED")
print("=" * 80)

print(
    "Reports:",
    REPORT_ROOT,
)

print(
    "Models:",
    MODEL_OUTPUT_ROOT,
)

DATASET
Train: (13984, 26)
Validation: (2997, 26)
Test: (2997, 26)

LEAKAGE-FREE FEATURE CONFIGURATION

Numeric:
   stars
   reviews_log1p
   bought_log1p
   isBestSeller
   cluster_id

Categorical:
   category_name

Excluded:
   price       -> target
   price_band  -> target leakage
   listPrice   -> excluded from clean baseline

FINAL FEATURE SHAPES
Train: (13984, 245)
Validation: (2997, 245)
Test: (2997, 245)

✅ price_band successfully removed from model features.
Total clean features: 245

TRAINING RANDOM FOREST

RandomForest - validation
------------------------------------------------------------
MAE:       28.5263
RMSE:      93.7904
Median AE: 13.5233
R²:        0.2928

RandomForest - test
------------------------------------------------------------
MAE:       27.7561
RMSE:      73.2382
Median AE: 13.5950
R²:        0.4030

TRAINING XGBOOST

XGBoost - validation
------------------------------------------------------------
MAE:       27.4634
RMSE:      95.6554
Median AE: 13.4802


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)



LightGBM - validation
------------------------------------------------------------
MAE:       26.5695
RMSE:      87.4034
Median AE: 12.7416
R²:        0.3858

LightGBM - test
------------------------------------------------------------
MAE:       26.1493
RMSE:      74.4214
Median AE: 12.3528
R²:        0.3835

LEAKAGE-FREE MODEL RANKING


,model,dataset,mae,rmse,median_absolute_error,r2
0,LightGBM,validation,26.569542,87.403418,12.741552,0.385813
1,XGBoost,validation,27.463381,95.655377,13.480227,0.264365
2,RandomForest,validation,28.526258,93.790383,13.523255,0.292771



BEST LEAKAGE-FREE MODEL
Model: LightGBM
Validation MAE: 26.5695
Validation RMSE: 87.4034
Validation Median AE: 12.7416
Validation R²: 0.3858

COMPLETED
Reports: /Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/data/reports/price_prediction/leakage_free
Models: /Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/models/price_prediction/leakage_free


# where LightGBM is failing. Run this error-analysis cell next

In [5]:
from pathlib import Path
import numpy as np
import pandas as pd


# ============================================================
# PATHS
# ============================================================

CURRENT_DIRECTORY = Path.cwd()

PROJECT_ROOT = (
    CURRENT_DIRECTORY.parent
    if CURRENT_DIRECTORY.name.lower() == "notebooks"
    else CURRENT_DIRECTORY
)

PREDICTION_FILE = (
    PROJECT_ROOT
    / "data"
    / "reports"
    / "price_prediction"
    / "leakage_free"
    / "clean_test_predictions.csv"
)


# ============================================================
# LOAD
# ============================================================

df = pd.read_csv(PREDICTION_FILE)

print("=" * 80)
print("LOADING CLEAN TEST PREDICTIONS")
print("=" * 80)

print("Rows:", f"{len(df):,}")
print("\nColumns:")
print(df.columns.tolist())


# ============================================================
# LIGHTGBM ERRORS
# ============================================================

df["prediction"] = df[
    "lightgbm_prediction"
]

df["error"] = (
    df["prediction"]
    - df["actual_price"]
)

df["absolute_error"] = (
    df["error"].abs()
)

df["squared_error"] = (
    df["error"] ** 2
)

df["percentage_error"] = (
    df["absolute_error"]
    / df["actual_price"].clip(lower=1)
    * 100
)


# ============================================================
# OVERALL
# ============================================================

print()
print("=" * 80)
print("OVERALL ERROR")
print("=" * 80)

print(
    "Mean Absolute Error:",
    round(df["absolute_error"].mean(), 4)
)

print(
    "Median Absolute Error:",
    round(df["absolute_error"].median(), 4)
)

print(
    "90th percentile error:",
    round(
        df["absolute_error"].quantile(0.90),
        4,
    )
)

print(
    "95th percentile error:",
    round(
        df["absolute_error"].quantile(0.95),
        4,
    )
)

print(
    "99th percentile error:",
    round(
        df["absolute_error"].quantile(0.99),
        4,
    )
)


# ============================================================
# PRICE BAND ANALYSIS
#
# price_band is allowed HERE because we are using it
# only to evaluate predictions after prediction.
# ============================================================

print()
print("=" * 80)
print("ERROR BY PRICE BAND")
print("=" * 80)

price_band_analysis = (
    df.groupby("price_band")
    .agg(
        products=("actual_price", "size"),
        average_price=("actual_price", "mean"),
        median_price=("actual_price", "median"),
        mae=("absolute_error", "mean"),
        median_ae=("absolute_error", "median"),
        rmse=(
            "squared_error",
            lambda x: np.sqrt(x.mean()),
        ),
        mean_percentage_error=(
            "percentage_error",
            "mean",
        ),
    )
    .sort_values("mae")
)

display(price_band_analysis)


# ============================================================
# CATEGORY ANALYSIS
# ============================================================

print()
print("=" * 80)
print("WORST CATEGORIES")
print("=" * 80)

category_analysis = (
    df.groupby("category_name")
    .agg(
        products=("actual_price", "size"),
        average_price=("actual_price", "mean"),
        mae=("absolute_error", "mean"),
        median_ae=("absolute_error", "median"),
        rmse=(
            "squared_error",
            lambda x: np.sqrt(x.mean()),
        ),
    )
)

# Ignore categories with extremely tiny test samples.
category_analysis_filtered = (
    category_analysis[
        category_analysis["products"] >= 10
    ]
    .sort_values(
        "mae",
        ascending=False,
    )
)

display(
    category_analysis_filtered.head(30)
)


# ============================================================
# CLUSTER ANALYSIS
# ============================================================

print()
print("=" * 80)
print("WORST CLUSTERS")
print("=" * 80)

cluster_analysis = (
    df.groupby("cluster_id")
    .agg(
        products=("actual_price", "size"),
        average_price=("actual_price", "mean"),
        mae=("absolute_error", "mean"),
        median_ae=("absolute_error", "median"),
        rmse=(
            "squared_error",
            lambda x: np.sqrt(x.mean()),
        ),
    )
    .sort_values(
        "mae",
        ascending=False,
    )
)

display(
    cluster_analysis.head(30)
)


# ============================================================
# WORST INDIVIDUAL PRODUCTS
# ============================================================

print()
print("=" * 80)
print("TOP 50 WORST PREDICTIONS")
print("=" * 80)

worst_predictions = (
    df.sort_values(
        "absolute_error",
        ascending=False,
    )
    .head(50)
)

display(
    worst_predictions[
        [
            "asin",
            "category_name",
            "price_band",
            "cluster_id",
            "actual_price",
            "prediction",
            "absolute_error",
            "percentage_error",
        ]
    ]
)


# ============================================================
# ERROR CONTRIBUTION
# ============================================================

df = df.sort_values(
    "squared_error",
    ascending=False,
).reset_index(drop=True)

df["cumulative_squared_error"] = (
    df["squared_error"].cumsum()
    / df["squared_error"].sum()
    * 100
)

print()
print("=" * 80)
print("OUTLIER ERROR CONTRIBUTION")
print("=" * 80)

for percentage in [
    0.01,
    0.05,
    0.10,
]:

    n = max(
        1,
        int(len(df) * percentage)
    )

    contribution = (
        df.iloc[:n]["squared_error"].sum()
        / df["squared_error"].sum()
        * 100
    )

    print(
        f"Worst {percentage * 100:.0f}% "
        f"of products contribute "
        f"{contribution:.2f}% of total squared error."
    )


# ============================================================
# SAVE ANALYSIS
# ============================================================

price_band_analysis.to_csv(
    PREDICTION_FILE.parent
    / "error_by_price_band.csv"
)

category_analysis_filtered.to_csv(
    PREDICTION_FILE.parent
    / "error_by_category.csv"
)

cluster_analysis.to_csv(
    PREDICTION_FILE.parent
    / "error_by_cluster.csv"
)

worst_predictions.to_csv(
    PREDICTION_FILE.parent
    / "worst_predictions.csv",
    index=False,
)

print()
print("=" * 80)
print("ERROR ANALYSIS COMPLETED")
print("=" * 80)

LOADING CLEAN TEST PREDICTIONS
Rows: 2,997

Columns:
['asin', 'actual_price', 'price_band', 'category_name', 'cluster_id', 'random_forest_prediction', 'xgboost_prediction', 'lightgbm_prediction']

OVERALL ERROR
Mean Absolute Error: 26.1493
Median Absolute Error: 12.3528
90th percentile error: 50.5953
95th percentile error: 84.301
99th percentile error: 274.2055

ERROR BY PRICE BAND


,products,average_price,median_price,mae,median_ae,rmse,mean_percentage_error
price_band,,,,,,,
medium,589,20.294024,19.990,8.898684,5.435975,31.204056,45.348816
low,558,13.791631,13.990,11.960126,10.800365,13.989974,88.358509
high,603,32.643466,30.950,13.879889,6.526438,23.427661,38.584399
very_low,509,8.436719,8.990,19.002862,15.931241,47.798625,240.785894
premium,524,65.165763,54.990,36.743783,23.424066,75.267129,52.617942
luxury,214,281.185374,189.995,136.254528,90.430454,231.349919,45.973451



WORST CATEGORIES


,products,average_price,mae,median_ae,rmse
category_name,,,,,
Computers & Tablets,22,484.600000,410.887958,277.911912,584.332121
Data Storage,11,100.530000,396.550455,624.371246,506.505178
Men's Watches,22,186.537727,119.283303,32.023672,227.389656
Computer Networking,15,170.783333,76.205354,17.582206,196.882064
Women's Watches,15,146.016000,73.945667,31.871741,118.949185
Measuring & Layout,12,65.062500,63.704942,33.692852,111.183220
Men's Shoes,39,103.719744,61.426122,55.699793,77.763701
Automotive Performance Parts & Accessories,20,138.643000,61.419959,48.095426,84.853750
Office Electronics,31,67.824839,59.822016,39.954749,85.914050



WORST CLUSTERS


,products,average_price,mae,median_ae,rmse
cluster_id,,,,,
55,13,516.675385,304.778729,273.846498,365.354731
31,30,537.122000,282.147772,188.631286,480.824422
36,15,244.161333,162.940814,99.691160,270.649964
81,18,230.380000,161.547116,76.230651,230.828540
39,30,118.896333,77.963572,58.107606,97.414726
15,10,83.834000,76.827597,38.160058,134.264715
92,36,81.448056,76.615828,36.613072,164.936649
25,15,109.032000,75.633405,50.961963,103.060208
9,39,159.406923,69.519892,69.021600,81.786138



TOP 50 WORST PREDICTIONS


,asin,category_name,price_band,cluster_id,actual_price,prediction,absolute_error,percentage_error
1639,B0BX75G9PN,Computers & Tablets,luxury,31,2196.99,316.934208,1880.055792,85.574162
2194,B08ZRTRLR9,Computers & Tablets,luxury,31,1499.00,324.369089,1174.630911,78.360968
1032,B0818BX4XG,Computers & Tablets,luxury,31,1376.00,476.435338,899.564662,65.375339
578,B09GHGKK59,Men's Watches,luxury,36,995.00,147.238842,847.761158,85.202126
435,B077BX86LM,Computer Networking,luxury,55,1263.11,514.797774,748.312226,59.243631
1588,B0BVRVRL95,Data Storage,very_low,66,9.99,728.450509,718.460509,7191.796884
2369,B002HWRJMQ,Data Storage,very_low,58,10.23,714.446305,704.216305,6883.834848
1510,B00AT4L934,Data Storage,medium,58,17.72,714.446305,696.726305,3931.864023
208,B088ZPBHHD,Data Storage,premium,92,69.41,737.386488,667.976488,962.363475
1047,B0B3LZZGWM,Data Storage,premium,92,73.99,737.386488,663.396488,896.602903



OUTLIER ERROR CONTRIBUTION
Worst 1% of products contribute 76.08% of total squared error.
Worst 5% of products contribute 91.65% of total squared error.
Worst 10% of products contribute 95.47% of total squared error.

ERROR ANALYSIS COMPLETED


In [7]:
import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    r2_score,
)


# ============================================================
# LOG TARGET
# ============================================================

y_train_log = np.log1p(y_train)
y_validation_log = np.log1p(y_validation)


print("=" * 80)
print("TRAINING LIGHTGBM WITH LOG1P(PRICE)")
print("=" * 80)


# ============================================================
# MODEL
# ============================================================

log_lgb_model = lgb.LGBMRegressor(
    n_estimators=3000,
    learning_rate=0.03,
    num_leaves=63,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.05,
    reg_lambda=1.0,
    random_state=42,
    verbosity=-1,
)


log_lgb_model.fit(
    X_train,
    y_train_log,

    eval_set=[
        (
            X_validation,
            y_validation_log,
        )
    ],

    callbacks=[
        lgb.early_stopping(
            100,
            verbose=False,
        )
    ],
)


# ============================================================
# PREDICT LOG PRICE
# ============================================================

validation_log_pred = (
    log_lgb_model.predict(
        X_validation
    )
)

test_log_pred = (
    log_lgb_model.predict(
        X_test
    )
)


# ============================================================
# CONVERT BACK TO REAL PRICE
# ============================================================

validation_pred = np.expm1(
    validation_log_pred
)

test_pred = np.expm1(
    test_log_pred
)

# Price cannot be negative
validation_pred = np.maximum(
    validation_pred,
    0,
)

test_pred = np.maximum(
    test_pred,
    0,
)


# ============================================================
# EVALUATION FUNCTION
# ============================================================

def evaluate_model(
    name,
    y_true,
    prediction,
):

    mae = mean_absolute_error(
        y_true,
        prediction,
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            prediction,
        )
    )

    median_ae = median_absolute_error(
        y_true,
        prediction,
    )

    r2 = r2_score(
        y_true,
        prediction,
    )

    print()
    print(name)
    print("-" * 60)

    print(
        f"MAE:       {mae:.4f}"
    )

    print(
        f"RMSE:      {rmse:.4f}"
    )

    print(
        f"Median AE: {median_ae:.4f}"
    )

    print(
        f"R²:        {r2:.4f}"
    )

    return {
        "model": name,
        "mae": mae,
        "rmse": rmse,
        "median_absolute_error": median_ae,
        "r2": r2,
    }


# ============================================================
# RESULTS
# ============================================================

validation_metrics = evaluate_model(
    "Log-LightGBM Validation",
    y_validation,
    validation_pred,
)

test_metrics = evaluate_model(
    "Log-LightGBM Test",
    y_test,
    test_pred,
)


# ============================================================
# DIRECT COMPARISON
# ============================================================

comparison = pd.DataFrame(
    [
        {
            "model":
                "Current Clean LightGBM",
            "validation_mae":
                26.5695,
            "validation_rmse":
                87.4034,
            "validation_median_ae":
                12.7416,
            "validation_r2":
                0.3858,
        },
        {
            "model":
                "Log1p LightGBM",
            "validation_mae":
                validation_metrics["mae"],
            "validation_rmse":
                validation_metrics["rmse"],
            "validation_median_ae":
                validation_metrics[
                    "median_absolute_error"
                ],
            "validation_r2":
                validation_metrics["r2"],
        },
    ]
)

print()
print("=" * 80)
print("DIRECT COMPARISON")
print("=" * 80)

display(comparison)


# ============================================================
# ERROR BY PRICE BAND
# ============================================================

analysis_df = test_df[
    [
        "asin",
        "category_name",
        "price_band",
        "cluster_id",
    ]
].copy()

analysis_df["actual_price"] = (
    y_test
)

analysis_df["prediction"] = (
    test_pred
)

analysis_df["absolute_error"] = (
    np.abs(
        analysis_df["actual_price"]
        - analysis_df["prediction"]
    )
)

analysis_df["squared_error"] = (
    (
        analysis_df["actual_price"]
        - analysis_df["prediction"]
    ) ** 2
)


band_results = (
    analysis_df
    .groupby("price_band")
    .agg(
        products=(
            "actual_price",
            "size",
        ),

        average_price=(
            "actual_price",
            "mean",
        ),

        mae=(
            "absolute_error",
            "mean",
        ),

        median_ae=(
            "absolute_error",
            "median",
        ),

        rmse=(
            "squared_error",
            lambda x:
                np.sqrt(x.mean()),
        ),
    )
    .sort_values("mae")
)


print()
print("=" * 80)
print("LOG MODEL ERROR BY PRICE BAND")
print("=" * 80)

display(
    band_results
)

TRAINING LIGHTGBM WITH LOG1P(PRICE)


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)



Log-LightGBM Validation
------------------------------------------------------------
MAE:       23.0321
RMSE:      85.0889
Median AE: 8.7942
R²:        0.4179

Log-LightGBM Test
------------------------------------------------------------
MAE:       22.8695
RMSE:      70.5412
Median AE: 8.6897
R²:        0.4461

DIRECT COMPARISON


,model,validation_mae,validation_rmse,validation_median_ae,validation_r2
0,Current Clean LightGBM,26.569500,87.403400,12.741600,0.385800
1,Log1p LightGBM,23.032105,85.088915,8.794242,0.417911



LOG MODEL ERROR BY PRICE BAND


,products,average_price,mae,median_ae,rmse
price_band,,,,,
medium,589,20.294024,5.354521,4.109086,7.767849
low,558,13.791631,5.422909,4.479646,7.164342
very_low,509,8.436719,9.019014,8.414515,10.099618
high,603,32.643467,13.895108,10.477111,18.723886
premium,524,65.165764,32.063930,25.715043,48.141365
luxury,214,281.185394,152.285720,99.842735,249.965880


# Title Feature Extraction + Log-LightGBM V2

In [8]:
from __future__ import annotations

import re
from pathlib import Path

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    r2_score,
)
from sklearn.pipeline import FeatureUnion
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# ============================================================
# CONFIGURATION
# ============================================================

CURRENT_DIRECTORY = Path.cwd()

PROJECT_ROOT = (
    CURRENT_DIRECTORY.parent
    if CURRENT_DIRECTORY.name.lower() == "notebooks"
    else CURRENT_DIRECTORY
)

MODEL_INPUT_ROOT = (
    PROJECT_ROOT
    / "data"
    / "amazon_multimodal"
    / "model_input"
)

MODEL_ROOT = (
    PROJECT_ROOT
    / "models"
    / "price_prediction"
    / "title_features_v2"
)

REPORT_ROOT = (
    PROJECT_ROOT
    / "data"
    / "reports"
    / "price_prediction"
    / "title_features_v2"
)

MODEL_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

REPORT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

RANDOM_STATE = 42


# ============================================================
# LOAD DATA
# ============================================================

train_df = pd.read_parquet(
    MODEL_INPUT_ROOT / "train.parquet"
)

validation_df = pd.read_parquet(
    MODEL_INPUT_ROOT / "validation.parquet"
)

test_df = pd.read_parquet(
    MODEL_INPUT_ROOT / "test.parquet"
)


print("=" * 80)
print("DATA LOADED")
print("=" * 80)

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)


# ============================================================
# TITLE FEATURE EXTRACTION
# ============================================================

def extract_first_number(text: str) -> float:
    matches = re.findall(
        r"\d+(?:\.\d+)?",
        str(text),
    )

    if not matches:
        return 0.0

    try:
        return float(matches[0])
    except Exception:
        return 0.0


def contains_pattern(
    text: str,
    pattern: str,
) -> int:
    return int(
        bool(
            re.search(
                pattern,
                str(text),
                flags=re.IGNORECASE,
            )
        )
    )


def create_engineered_features(
    df: pd.DataFrame,
) -> pd.DataFrame:

    df = df.copy()

    # --------------------------------------------------------
    # BASE CLEANING
    # --------------------------------------------------------

    df["title"] = (
        df["title"]
        .fillna("")
        .astype(str)
    )

    df["category_name"] = (
        df["category_name"]
        .fillna("Unknown")
        .astype(str)
    )

    df["stars"] = pd.to_numeric(
        df["stars"],
        errors="coerce",
    ).fillna(0)

    df["reviews"] = pd.to_numeric(
        df["reviews"],
        errors="coerce",
    ).fillna(0)

    df["boughtInLastMonth"] = (
        pd.to_numeric(
            df["boughtInLastMonth"],
            errors="coerce",
        )
        .fillna(0)
    )

    df["isBestSeller"] = (
        df["isBestSeller"]
        .fillna(False)
        .astype(int)
    )

    df["cluster_id"] = (
        pd.to_numeric(
            df["cluster_id"],
            errors="coerce",
        )
        .fillna(-1)
    )

    # --------------------------------------------------------
    # LOG FEATURES
    # --------------------------------------------------------

    df["reviews_log1p"] = np.log1p(
        df["reviews"]
    )

    df["bought_log1p"] = np.log1p(
        df["boughtInLastMonth"]
    )

    # --------------------------------------------------------
    # TITLE LENGTH FEATURES
    # --------------------------------------------------------

    df["title_char_length"] = (
        df["title"].str.len()
    )

    df["title_word_count"] = (
        df["title"]
        .str.split()
        .str.len()
        .fillna(0)
    )

    df["title_digit_count"] = (
        df["title"]
        .str.count(r"\d")
    )

    df["title_uppercase_count"] = (
        df["title"]
        .apply(
            lambda text: sum(
                1
                for char in text
                if char.isupper()
            )
        )
    )

    # --------------------------------------------------------
    # FIRST NUMERIC VALUE IN TITLE
    # --------------------------------------------------------

    df["title_first_number"] = (
        df["title"]
        .apply(extract_first_number)
    )

    # --------------------------------------------------------
    # CAPACITY / STORAGE FEATURES
    # --------------------------------------------------------

    df["has_gb"] = (
        df["title"]
        .apply(
            lambda x:
            contains_pattern(
                x,
                r"\b\d+(?:\.\d+)?\s*gb\b",
            )
        )
    )

    df["has_tb"] = (
        df["title"]
        .apply(
            lambda x:
            contains_pattern(
                x,
                r"\b\d+(?:\.\d+)?\s*tb\b",
            )
        )
    )

    # --------------------------------------------------------
    # MEMORY FEATURES
    # --------------------------------------------------------

    df["has_ram"] = (
        df["title"]
        .apply(
            lambda x:
            contains_pattern(
                x,
                r"\b(?:ram|memory)\b",
            )
        )
    )

    # --------------------------------------------------------
    # SIZE FEATURES
    # --------------------------------------------------------

    df["has_inch"] = (
        df["title"]
        .apply(
            lambda x:
            contains_pattern(
                x,
                r"\b\d+(?:\.\d+)?\s*(?:inch|inches|\")",
            )
        )
    )

    df["has_cm"] = (
        df["title"]
        .apply(
            lambda x:
            contains_pattern(
                x,
                r"\b\d+(?:\.\d+)?\s*cm\b",
            )
        )
    )

    # --------------------------------------------------------
    # WEIGHT FEATURES
    # --------------------------------------------------------

    df["has_kg"] = (
        df["title"]
        .apply(
            lambda x:
            contains_pattern(
                x,
                r"\b\d+(?:\.\d+)?\s*kg\b",
            )
        )
    )

    df["has_gram"] = (
        df["title"]
        .apply(
            lambda x:
            contains_pattern(
                x,
                r"\b\d+(?:\.\d+)?\s*(?:g|gram|grams)\b",
            )
        )
    )

    # --------------------------------------------------------
    # POWER FEATURES
    # --------------------------------------------------------

    df["has_watt"] = (
        df["title"]
        .apply(
            lambda x:
            contains_pattern(
                x,
                r"\b\d+(?:\.\d+)?\s*(?:w|watt|watts)\b",
            )
        )
    )

    df["has_volt"] = (
        df["title"]
        .apply(
            lambda x:
            contains_pattern(
                x,
                r"\b\d+(?:\.\d+)?\s*(?:v|volt|volts)\b",
            )
        )
    )

    # --------------------------------------------------------
    # QUANTITY FEATURES
    # --------------------------------------------------------

    df["has_pack"] = (
        df["title"]
        .apply(
            lambda x:
            contains_pattern(
                x,
                r"\b(?:pack|set|pair|bundle)\b",
            )
        )
    )

    df["has_multipack_number"] = (
        df["title"]
        .apply(
            lambda x:
            contains_pattern(
                x,
                r"\b\d+\s*[- ]?(?:pack|piece|pcs|count|ct)\b",
            )
        )
    )

    # --------------------------------------------------------
    # COMMERCIAL TITLE SIGNALS
    # --------------------------------------------------------

    df["has_pro"] = (
        df["title"]
        .apply(
            lambda x:
            contains_pattern(
                x,
                r"\bpro\b",
            )
        )
    )

    df["has_max"] = (
        df["title"]
        .apply(
            lambda x:
            contains_pattern(
                x,
                r"\bmax\b",
            )
        )
    )

    df["has_premium"] = (
        df["title"]
        .apply(
            lambda x:
            contains_pattern(
                x,
                r"\bpremium\b",
            )
        )
    )

    df["has_professional"] = (
        df["title"]
        .apply(
            lambda x:
            contains_pattern(
                x,
                r"\bprofessional\b",
            )
        )
    )

    df["has_wireless"] = (
        df["title"]
        .apply(
            lambda x:
            contains_pattern(
                x,
                r"\bwireless\b",
            )
        )
    )

    df["has_smart"] = (
        df["title"]
        .apply(
            lambda x:
            contains_pattern(
                x,
                r"\bsmart\b",
            )
        )
    )

    return df


train_df = create_engineered_features(
    train_df
)

validation_df = (
    create_engineered_features(
        validation_df
    )
)

test_df = create_engineered_features(
    test_df
)


# ============================================================
# FEATURES
# ============================================================

NUMERIC_FEATURES = [
    "stars",
    "reviews_log1p",
    "bought_log1p",
    "isBestSeller",
    "cluster_id",

    "title_char_length",
    "title_word_count",
    "title_digit_count",
    "title_uppercase_count",
    "title_first_number",

    "has_gb",
    "has_tb",
    "has_ram",

    "has_inch",
    "has_cm",

    "has_kg",
    "has_gram",

    "has_watt",
    "has_volt",

    "has_pack",
    "has_multipack_number",

    "has_pro",
    "has_max",
    "has_premium",
    "has_professional",
    "has_wireless",
    "has_smart",
]

CATEGORICAL_FEATURES = [
    "category_name",
]


# ============================================================
# STRUCTURED PREPROCESSOR
# ============================================================

structured_preprocessor = (
    ColumnTransformer(
        transformers=[
            (
                "numeric",
                StandardScaler(),
                NUMERIC_FEATURES,
            ),
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
                CATEGORICAL_FEATURES,
            ),
        ]
    )
)


X_train_structured = (
    structured_preprocessor
    .fit_transform(train_df)
    .astype(np.float32)
)

X_validation_structured = (
    structured_preprocessor
    .transform(validation_df)
    .astype(np.float32)
)

X_test_structured = (
    structured_preprocessor
    .transform(test_df)
    .astype(np.float32)
)


# ============================================================
# TF-IDF TITLE FEATURES
# ============================================================

tfidf = TfidfVectorizer(
    lowercase=True,

    strip_accents="unicode",

    ngram_range=(1, 2),

    min_df=3,

    max_df=0.98,

    max_features=8000,

    sublinear_tf=True,

    dtype=np.float32,
)


X_train_title = (
    tfidf.fit_transform(
        train_df["title"]
    )
)

X_validation_title = (
    tfidf.transform(
        validation_df["title"]
    )
)

X_test_title = (
    tfidf.transform(
        test_df["title"]
    )
)


print()
print("=" * 80)
print("FEATURE SHAPES")
print("=" * 80)

print(
    "Structured:",
    X_train_structured.shape
)

print(
    "TF-IDF title:",
    X_train_title.shape
)


# ============================================================
# COMBINE FEATURES
# ============================================================

from scipy.sparse import (
    csr_matrix,
    hstack,
)


X_train = hstack(
    [
        csr_matrix(
            X_train_structured
        ),
        X_train_title,
    ],
    format="csr",
)

X_validation = hstack(
    [
        csr_matrix(
            X_validation_structured
        ),
        X_validation_title,
    ],
    format="csr",
)

X_test = hstack(
    [
        csr_matrix(
            X_test_structured
        ),
        X_test_title,
    ],
    format="csr",
)


print(
    "Combined train shape:",
    X_train.shape
)


# ============================================================
# TARGET
# ============================================================

y_train = (
    train_df["price"]
    .astype(np.float32)
    .to_numpy()
)

y_validation = (
    validation_df["price"]
    .astype(np.float32)
    .to_numpy()
)

y_test = (
    test_df["price"]
    .astype(np.float32)
    .to_numpy()
)


# ============================================================
# LOG TARGET
# ============================================================

y_train_log = np.log1p(
    y_train
)

y_validation_log = np.log1p(
    y_validation
)


# ============================================================
# TRAIN LIGHTGBM
# ============================================================

print()
print("=" * 80)
print(
    "TRAINING TITLE-ENRICHED "
    "LOG-LIGHTGBM"
)
print("=" * 80)


model = lgb.LGBMRegressor(
    objective="regression",

    n_estimators=4000,

    learning_rate=0.025,

    num_leaves=63,

    max_depth=-1,

    min_child_samples=20,

    subsample=0.85,

    colsample_bytree=0.85,

    reg_alpha=0.05,

    reg_lambda=1.0,

    random_state=RANDOM_STATE,

    verbosity=-1,

    n_jobs=-1,
)


model.fit(
    X_train,
    y_train_log,

    eval_set=[
        (
            X_validation,
            y_validation_log,
        )
    ],

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=150,
            verbose=False,
        )
    ],
)


# ============================================================
# PREDICTIONS
# ============================================================

validation_log_prediction = (
    model.predict(
        X_validation
    )
)

test_log_prediction = (
    model.predict(
        X_test
    )
)


validation_prediction = np.expm1(
    validation_log_prediction
)

test_prediction = np.expm1(
    test_log_prediction
)

validation_prediction = np.clip(
    validation_prediction,
    0,
    None,
)

test_prediction = np.clip(
    test_prediction,
    0,
    None,
)


# ============================================================
# METRICS
# ============================================================

def evaluate(
    name,
    y_true,
    prediction,
):

    mae = mean_absolute_error(
        y_true,
        prediction,
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            prediction,
        )
    )

    median_ae = (
        median_absolute_error(
            y_true,
            prediction,
        )
    )

    r2 = r2_score(
        y_true,
        prediction,
    )

    print()
    print(name)
    print("-" * 60)

    print(
        f"MAE:       {mae:.4f}"
    )

    print(
        f"RMSE:      {rmse:.4f}"
    )

    print(
        f"Median AE: {median_ae:.4f}"
    )

    print(
        f"R²:        {r2:.4f}"
    )

    return {
        "mae": mae,
        "rmse": rmse,
        "median_ae": median_ae,
        "r2": r2,
    }


validation_metrics = evaluate(
    "Validation",
    y_validation,
    validation_prediction,
)

test_metrics = evaluate(
    "Test",
    y_test,
    test_prediction,
)


# ============================================================
# COMPARE AGAINST CURRENT BEST CLEAN MODEL
# ============================================================

comparison_df = pd.DataFrame(
    [
        {
            "model":
                "Clean Log-LightGBM",

            "validation_mae":
                23.0321,

            "validation_rmse":
                85.0889,

            "validation_median_ae":
                8.7942,

            "validation_r2":
                0.4179,
        },

        {
            "model":
                "Title-Enriched Log-LightGBM",

            "validation_mae":
                validation_metrics["mae"],

            "validation_rmse":
                validation_metrics["rmse"],

            "validation_median_ae":
                validation_metrics[
                    "median_ae"
                ],

            "validation_r2":
                validation_metrics["r2"],
        },
    ]
)


print()
print("=" * 80)
print("MODEL COMPARISON")
print("=" * 80)

display(
    comparison_df
)


# ============================================================
# PRICE BAND ANALYSIS
# ============================================================

analysis_df = test_df[
    [
        "asin",
        "title",
        "category_name",
        "price_band",
        "cluster_id",
    ]
].copy()

analysis_df[
    "actual_price"
] = y_test

analysis_df[
    "predicted_price"
] = test_prediction

analysis_df[
    "absolute_error"
] = np.abs(
    y_test
    - test_prediction
)

analysis_df[
    "squared_error"
] = (
    y_test
    - test_prediction
) ** 2


band_analysis = (
    analysis_df
    .groupby(
        "price_band"
    )
    .agg(
        products=(
            "actual_price",
            "size",
        ),

        mean_price=(
            "actual_price",
            "mean",
        ),

        mae=(
            "absolute_error",
            "mean",
        ),

        median_ae=(
            "absolute_error",
            "median",
        ),

        rmse=(
            "squared_error",
            lambda x:
                np.sqrt(
                    x.mean()
                ),
        ),
    )
    .sort_values("mae")
)


print()
print("=" * 80)
print("ERROR BY PRICE BAND")
print("=" * 80)

display(
    band_analysis
)


# ============================================================
# SAVE ARTIFACTS
# ============================================================

joblib.dump(
    model,
    MODEL_ROOT
    / "title_enriched_log_lightgbm.joblib",
)

joblib.dump(
    structured_preprocessor,
    MODEL_ROOT
    / "structured_preprocessor.joblib",
)

joblib.dump(
    tfidf,
    MODEL_ROOT
    / "title_tfidf.joblib",
)


comparison_df.to_csv(
    REPORT_ROOT
    / "model_comparison.csv",
    index=False,
)

band_analysis.to_csv(
    REPORT_ROOT
    / "error_by_price_band.csv",
)

analysis_df.to_csv(
    REPORT_ROOT
    / "test_predictions.csv",
    index=False,
)


print()
print("=" * 80)
print("TITLE-ENRICHED MODEL COMPLETED")
print("=" * 80)

print(
    "Model saved:",
    MODEL_ROOT
)

print(
    "Reports saved:",
    REPORT_ROOT
)

DATA LOADED
Train: (13984, 26)
Validation: (2997, 26)
Test: (2997, 26)

FEATURE SHAPES
Structured: (13984, 267)
TF-IDF title: (13984, 8000)
Combined train shape: (13984, 8267)

TRAINING TITLE-ENRICHED LOG-LIGHTGBM


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)



Validation
------------------------------------------------------------
MAE:       21.9827
RMSE:      85.2617
Median AE: 7.9673
R²:        0.4155

Test
------------------------------------------------------------
MAE:       21.6772
RMSE:      65.0531
Median AE: 7.9357
R²:        0.5290

MODEL COMPARISON


,model,validation_mae,validation_rmse,validation_median_ae,validation_r2
0,Clean Log-LightGBM,23.032100,85.088900,8.794200,0.417900
1,Title-Enriched Log-LightGBM,21.982725,85.261717,7.967319,0.415544



ERROR BY PRICE BAND


,products,mean_price,mae,median_ae,rmse
price_band,,,,,
low,558,13.791631,5.482254,4.035812,7.591052
medium,589,20.294024,5.856881,4.334546,8.849987
very_low,509,8.436719,8.159980,7.148863,10.097107
high,603,32.643467,12.038568,9.485109,16.246133
premium,524,65.165764,27.509142,22.460349,38.907795
luxury,214,281.185394,152.477581,102.309535,232.825618



TITLE-ENRICHED MODEL COMPLETED
Model saved: /Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/models/price_prediction/title_features_v2
Reports saved: /Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/data/reports/price_prediction/title_features_v2


## Now test whether the saved 512-D CLIP text embeddings add useful signal beyond TF-IDF and structured features.

In [9]:
from __future__ import annotations

from pathlib import Path

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd

from scipy.sparse import csr_matrix, hstack
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    r2_score,
)
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# ============================================================
# CONFIG
# ============================================================

CURRENT_DIRECTORY = Path.cwd()

PROJECT_ROOT = (
    CURRENT_DIRECTORY.parent
    if CURRENT_DIRECTORY.name.lower() == "notebooks"
    else CURRENT_DIRECTORY
)

MODEL_INPUT_ROOT = (
    PROJECT_ROOT
    / "data"
    / "amazon_multimodal"
    / "model_input"
)

EMBEDDING_ROOT = (
    PROJECT_ROOT
    / "data"
    / "amazon_multimodal"
    / "embeddings"
    / "clip"
)

MODEL_ROOT = (
    PROJECT_ROOT
    / "models"
    / "price_prediction"
    / "clip_text_ablation"
)

REPORT_ROOT = (
    PROJECT_ROOT
    / "data"
    / "reports"
    / "price_prediction"
    / "clip_text_ablation"
)

MODEL_ROOT.mkdir(parents=True, exist_ok=True)
REPORT_ROOT.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42


# ============================================================
# LOAD DATA
# ============================================================

train_df = pd.read_parquet(
    MODEL_INPUT_ROOT / "train.parquet"
)

validation_df = pd.read_parquet(
    MODEL_INPUT_ROOT / "validation.parquet"
)

test_df = pd.read_parquet(
    MODEL_INPUT_ROOT / "test.parquet"
)

X_train_clip = np.load(
    EMBEDDING_ROOT / "train_text_embeddings.npy"
).astype(np.float32)

X_validation_clip = np.load(
    EMBEDDING_ROOT / "validation_text_embeddings.npy"
).astype(np.float32)

X_test_clip = np.load(
    EMBEDDING_ROOT / "test_text_embeddings.npy"
).astype(np.float32)


print("Train rows:", len(train_df))
print("CLIP train shape:", X_train_clip.shape)


# ============================================================
# BASE FEATURE ENGINEERING
# ============================================================

def create_features(df):
    df = df.copy()

    df["title"] = (
        df["title"]
        .fillna("")
        .astype(str)
    )

    df["category_name"] = (
        df["category_name"]
        .fillna("Unknown")
        .astype(str)
    )

    df["stars"] = pd.to_numeric(
        df["stars"],
        errors="coerce",
    ).fillna(0)

    df["reviews_log1p"] = np.log1p(
        pd.to_numeric(
            df["reviews"],
            errors="coerce",
        ).fillna(0)
    )

    df["bought_log1p"] = np.log1p(
        pd.to_numeric(
            df["boughtInLastMonth"],
            errors="coerce",
        ).fillna(0)
    )

    df["isBestSeller"] = (
        df["isBestSeller"]
        .fillna(False)
        .astype(int)
    )

    df["cluster_id"] = pd.to_numeric(
        df["cluster_id"],
        errors="coerce",
    ).fillna(-1)

    return df


train_df = create_features(train_df)
validation_df = create_features(validation_df)
test_df = create_features(test_df)


# ============================================================
# STRUCTURED FEATURES
# ============================================================

NUMERIC_FEATURES = [
    "stars",
    "reviews_log1p",
    "bought_log1p",
    "isBestSeller",
    "cluster_id",
]

CATEGORICAL_FEATURES = [
    "category_name",
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            NUMERIC_FEATURES,
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
            CATEGORICAL_FEATURES,
        ),
    ]
)

X_train_structured = (
    preprocessor
    .fit_transform(train_df)
    .astype(np.float32)
)

X_validation_structured = (
    preprocessor
    .transform(validation_df)
    .astype(np.float32)
)

X_test_structured = (
    preprocessor
    .transform(test_df)
    .astype(np.float32)
)


# ============================================================
# TF-IDF
# ============================================================

tfidf = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.98,
    max_features=8000,
    sublinear_tf=True,
    dtype=np.float32,
)

X_train_tfidf = tfidf.fit_transform(
    train_df["title"]
)

X_validation_tfidf = tfidf.transform(
    validation_df["title"]
)

X_test_tfidf = tfidf.transform(
    test_df["title"]
)


# ============================================================
# TARGET
# ============================================================

y_train = train_df["price"].astype(np.float32).to_numpy()
y_validation = validation_df["price"].astype(np.float32).to_numpy()
y_test = test_df["price"].astype(np.float32).to_numpy()

y_train_log = np.log1p(y_train)
y_validation_log = np.log1p(y_validation)


# ============================================================
# EXPERIMENT FEATURE SETS
# ============================================================

feature_sets = {
    "structured_plus_clip": (
        hstack(
            [
                csr_matrix(X_train_structured),
                csr_matrix(X_train_clip),
            ],
            format="csr",
        ),
        hstack(
            [
                csr_matrix(X_validation_structured),
                csr_matrix(X_validation_clip),
            ],
            format="csr",
        ),
        hstack(
            [
                csr_matrix(X_test_structured),
                csr_matrix(X_test_clip),
            ],
            format="csr",
        ),
    ),

    "structured_tfidf_clip": (
        hstack(
            [
                csr_matrix(X_train_structured),
                X_train_tfidf,
                csr_matrix(X_train_clip),
            ],
            format="csr",
        ),
        hstack(
            [
                csr_matrix(X_validation_structured),
                X_validation_tfidf,
                csr_matrix(X_validation_clip),
            ],
            format="csr",
        ),
        hstack(
            [
                csr_matrix(X_test_structured),
                X_test_tfidf,
                csr_matrix(X_test_clip),
            ],
            format="csr",
        ),
    ),
}


# ============================================================
# METRIC FUNCTION
# ============================================================

def evaluate(y_true, prediction):
    return {
        "mae": mean_absolute_error(
            y_true,
            prediction,
        ),
        "rmse": np.sqrt(
            mean_squared_error(
                y_true,
                prediction,
            )
        ),
        "median_ae": median_absolute_error(
            y_true,
            prediction,
        ),
        "r2": r2_score(
            y_true,
            prediction,
        ),
    }


# ============================================================
# TRAIN EXPERIMENTS
# ============================================================

results = []

for experiment_name, (
    X_train,
    X_validation,
    X_test,
) in feature_sets.items():

    print()
    print("=" * 80)
    print(
        "EXPERIMENT:",
        experiment_name
    )
    print("=" * 80)

    print(
        "Train feature shape:",
        X_train.shape
    )

    model = lgb.LGBMRegressor(
        objective="regression",
        n_estimators=4000,
        learning_rate=0.025,
        num_leaves=63,
        max_depth=-1,
        min_child_samples=20,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.05,
        reg_lambda=1.0,
        random_state=RANDOM_STATE,
        verbosity=-1,
        n_jobs=-1,
    )

    model.fit(
        X_train,
        y_train_log,
        eval_set=[
            (
                X_validation,
                y_validation_log,
            )
        ],
        callbacks=[
            lgb.early_stopping(
                stopping_rounds=150,
                verbose=False,
            )
        ],
    )

    validation_prediction = np.expm1(
        model.predict(
            X_validation
        )
    )

    test_prediction = np.expm1(
        model.predict(
            X_test
        )
    )

    validation_prediction = np.clip(
        validation_prediction,
        0,
        None,
    )

    test_prediction = np.clip(
        test_prediction,
        0,
        None,
    )

    validation_metrics = evaluate(
        y_validation,
        validation_prediction,
    )

    test_metrics = evaluate(
        y_test,
        test_prediction,
    )

    print(
        f"Validation MAE: "
        f"{validation_metrics['mae']:.4f}"
    )

    print(
        f"Validation RMSE: "
        f"{validation_metrics['rmse']:.4f}"
    )

    print(
        f"Validation Median AE: "
        f"{validation_metrics['median_ae']:.4f}"
    )

    print(
        f"Validation R²: "
        f"{validation_metrics['r2']:.4f}"
    )

    results.append(
        {
            "experiment": experiment_name,
            "validation_mae":
                validation_metrics["mae"],
            "validation_rmse":
                validation_metrics["rmse"],
            "validation_median_ae":
                validation_metrics["median_ae"],
            "validation_r2":
                validation_metrics["r2"],
            "test_mae":
                test_metrics["mae"],
            "test_rmse":
                test_metrics["rmse"],
            "test_median_ae":
                test_metrics["median_ae"],
            "test_r2":
                test_metrics["r2"],
        }
    )

    joblib.dump(
        model,
        MODEL_ROOT
        / f"{experiment_name}.joblib",
    )


# ============================================================
# COMPARE WITH CURRENT BEST
# ============================================================

results_df = pd.DataFrame(
    [
        {
            "experiment":
                "title_enriched_tfidf",
            "validation_mae":
                21.9827,
            "validation_rmse":
                85.2617,
            "validation_median_ae":
                7.9673,
            "validation_r2":
                0.4155,
            "test_mae":
                21.6772,
            "test_rmse":
                65.0531,
            "test_median_ae":
                7.9357,
            "test_r2":
                0.5290,
        },
        *results,
    ]
)

results_df = (
    results_df
    .sort_values(
        "validation_mae"
    )
    .reset_index(drop=True)
)

print()
print("=" * 80)
print("CLIP TEXT ABLATION RESULTS")
print("=" * 80)

display(results_df)


results_df.to_csv(
    REPORT_ROOT
    / "clip_text_ablation_results.csv",
    index=False,
)


joblib.dump(
    preprocessor,
    MODEL_ROOT
    / "structured_preprocessor.joblib",
)

joblib.dump(
    tfidf,
    MODEL_ROOT
    / "title_tfidf.joblib",
)

print()
print("✅ CLIP text ablation completed.")

Train rows: 13984
CLIP train shape: (13984, 512)

EXPERIMENT: structured_plus_clip
Train feature shape: (13984, 757)


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Validation MAE: 23.2051
Validation RMSE: 93.6505
Validation Median AE: 7.7062
Validation R²: 0.2949

EXPERIMENT: structured_tfidf_clip
Train feature shape: (13984, 8757)


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Validation MAE: 22.9843
Validation RMSE: 92.6790
Validation Median AE: 7.6516
Validation R²: 0.3094

CLIP TEXT ABLATION RESULTS


,experiment,validation_mae,validation_rmse,validation_median_ae,validation_r2,test_mae,test_rmse,test_median_ae,test_r2
0,title_enriched_tfidf,21.982700,85.261700,7.967300,0.415500,21.677200,65.053100,7.935700,0.529000
1,structured_tfidf_clip,22.984292,92.679019,7.651645,0.309432,22.346267,70.959577,7.537906,0.439534
2,structured_plus_clip,23.205063,93.650495,7.706161,0.294879,22.668927,72.726750,7.713700,0.411270



✅ CLIP text ablation completed.


## the cluster ablation now. We will keep the current best architecture—structured features + engineered title features + TF-IDF + log1p(price)

In [10]:
from __future__ import annotations

import re
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd

from scipy.sparse import csr_matrix, hstack

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    r2_score,
)
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)


# ============================================================
# CONFIGURATION
# ============================================================

CURRENT_DIRECTORY = Path.cwd()

PROJECT_ROOT = (
    CURRENT_DIRECTORY.parent
    if CURRENT_DIRECTORY.name.lower() == "notebooks"
    else CURRENT_DIRECTORY
)

MODEL_INPUT_ROOT = (
    PROJECT_ROOT
    / "data"
    / "amazon_multimodal"
    / "model_input"
)

REPORT_ROOT = (
    PROJECT_ROOT
    / "data"
    / "reports"
    / "price_prediction"
    / "cluster_ablation"
)

REPORT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

RANDOM_STATE = 42


# ============================================================
# LOAD DATA
# ============================================================

train_df = pd.read_parquet(
    MODEL_INPUT_ROOT / "train.parquet"
)

validation_df = pd.read_parquet(
    MODEL_INPUT_ROOT / "validation.parquet"
)

test_df = pd.read_parquet(
    MODEL_INPUT_ROOT / "test.parquet"
)


print("=" * 80)
print("DATA")
print("=" * 80)

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)


# ============================================================
# HELPERS
# ============================================================

def contains_pattern(text, pattern):

    return int(
        bool(
            re.search(
                pattern,
                str(text),
                flags=re.IGNORECASE,
            )
        )
    )


def extract_first_number(text):

    numbers = re.findall(
        r"\d+(?:\.\d+)?",
        str(text),
    )

    if not numbers:
        return 0.0

    try:
        return float(numbers[0])

    except Exception:
        return 0.0


# ============================================================
# FEATURE ENGINEERING
# ============================================================

def create_features(df):

    df = df.copy()

    df["title"] = (
        df["title"]
        .fillna("")
        .astype(str)
    )

    df["category_name"] = (
        df["category_name"]
        .fillna("Unknown")
        .astype(str)
    )

    df["stars"] = pd.to_numeric(
        df["stars"],
        errors="coerce",
    ).fillna(0)

    reviews = pd.to_numeric(
        df["reviews"],
        errors="coerce",
    ).fillna(0)

    bought = pd.to_numeric(
        df["boughtInLastMonth"],
        errors="coerce",
    ).fillna(0)

    df["reviews_log1p"] = np.log1p(
        reviews
    )

    df["bought_log1p"] = np.log1p(
        bought
    )

    df["isBestSeller"] = (
        df["isBestSeller"]
        .fillna(False)
        .astype(int)
    )

    df["cluster_id"] = pd.to_numeric(
        df["cluster_id"],
        errors="coerce",
    ).fillna(-1)

    # ========================================================
    # TITLE STATISTICS
    # ========================================================

    df["title_char_length"] = (
        df["title"].str.len()
    )

    df["title_word_count"] = (
        df["title"]
        .str.split()
        .str.len()
        .fillna(0)
    )

    df["title_digit_count"] = (
        df["title"]
        .str.count(r"\d")
    )

    df["title_uppercase_count"] = (
        df["title"].apply(
            lambda text:
            sum(
                character.isupper()
                for character in text
            )
        )
    )

    df["title_first_number"] = (
        df["title"]
        .apply(extract_first_number)
    )

    # ========================================================
    # PRODUCT ATTRIBUTE SIGNALS
    # ========================================================

    patterns = {

        "has_gb":
            r"\b\d+(?:\.\d+)?\s*gb\b",

        "has_tb":
            r"\b\d+(?:\.\d+)?\s*tb\b",

        "has_ram":
            r"\b(?:ram|memory)\b",

        "has_inch":
            r'\b\d+(?:\.\d+)?\s*(?:inch|inches|")',

        "has_cm":
            r"\b\d+(?:\.\d+)?\s*cm\b",

        "has_kg":
            r"\b\d+(?:\.\d+)?\s*kg\b",

        "has_gram":
            r"\b\d+(?:\.\d+)?\s*(?:g|gram|grams)\b",

        "has_watt":
            r"\b\d+(?:\.\d+)?\s*(?:w|watt|watts)\b",

        "has_volt":
            r"\b\d+(?:\.\d+)?\s*(?:v|volt|volts)\b",

        "has_pack":
            r"\b(?:pack|set|pair|bundle)\b",

        "has_multipack_number":
            r"\b\d+\s*[- ]?(?:pack|piece|pcs|count|ct)\b",

        "has_pro":
            r"\bpro\b",

        "has_max":
            r"\bmax\b",

        "has_premium":
            r"\bpremium\b",

        "has_professional":
            r"\bprofessional\b",

        "has_wireless":
            r"\bwireless\b",

        "has_smart":
            r"\bsmart\b",
    }

    for feature_name, pattern in patterns.items():

        df[feature_name] = (
            df["title"]
            .apply(
                lambda text:
                contains_pattern(
                    text,
                    pattern,
                )
            )
        )

    return df


train_df = create_features(train_df)

validation_df = create_features(
    validation_df
)

test_df = create_features(
    test_df
)


# ============================================================
# BASE FEATURES
# ============================================================

BASE_NUMERIC_FEATURES = [

    "stars",
    "reviews_log1p",
    "bought_log1p",
    "isBestSeller",

    "title_char_length",
    "title_word_count",
    "title_digit_count",
    "title_uppercase_count",
    "title_first_number",

    "has_gb",
    "has_tb",
    "has_ram",

    "has_inch",
    "has_cm",

    "has_kg",
    "has_gram",

    "has_watt",
    "has_volt",

    "has_pack",
    "has_multipack_number",

    "has_pro",
    "has_max",
    "has_premium",
    "has_professional",
    "has_wireless",
    "has_smart",
]

CATEGORICAL_FEATURES = [
    "category_name",
]


# ============================================================
# TF-IDF
# ============================================================

print()
print("=" * 80)
print("BUILDING TF-IDF")
print("=" * 80)


tfidf = TfidfVectorizer(

    lowercase=True,

    strip_accents="unicode",

    ngram_range=(1, 2),

    min_df=3,

    max_df=0.98,

    max_features=8000,

    sublinear_tf=True,

    dtype=np.float32,
)


X_train_tfidf = tfidf.fit_transform(
    train_df["title"]
)

X_validation_tfidf = tfidf.transform(
    validation_df["title"]
)

X_test_tfidf = tfidf.transform(
    test_df["title"]
)


print(
    "TF-IDF:",
    X_train_tfidf.shape
)


# ============================================================
# TARGET
# ============================================================

y_train = (
    train_df["price"]
    .astype(np.float32)
    .to_numpy()
)

y_validation = (
    validation_df["price"]
    .astype(np.float32)
    .to_numpy()
)

y_test = (
    test_df["price"]
    .astype(np.float32)
    .to_numpy()
)


y_train_log = np.log1p(
    y_train
)

y_validation_log = np.log1p(
    y_validation
)


# ============================================================
# EVALUATION
# ============================================================

def evaluate(
    y_true,
    prediction,
):

    return {

        "mae":
            mean_absolute_error(
                y_true,
                prediction,
            ),

        "rmse":
            np.sqrt(
                mean_squared_error(
                    y_true,
                    prediction,
                )
            ),

        "median_ae":
            median_absolute_error(
                y_true,
                prediction,
            ),

        "r2":
            r2_score(
                y_true,
                prediction,
            ),
    }


# ============================================================
# EXPERIMENT FUNCTION
# ============================================================

def run_experiment(
    experiment_name,
    use_cluster,
):

    print()
    print("=" * 80)

    print(
        "EXPERIMENT:",
        experiment_name
    )

    print("=" * 80)


    # ========================================================
    # SELECT FEATURES
    # ========================================================

    numeric_features = (
        BASE_NUMERIC_FEATURES.copy()
    )

    if use_cluster:

        numeric_features.append(
            "cluster_id"
        )


    print(
        "Using cluster_id:",
        use_cluster
    )

    print(
        "Numeric features:",
        len(numeric_features)
    )


    # ========================================================
    # PREPROCESSOR
    # ========================================================

    preprocessor = (
        ColumnTransformer(
            transformers=[
                (
                    "numeric",
                    StandardScaler(),
                    numeric_features,
                ),

                (
                    "categorical",
                    OneHotEncoder(
                        handle_unknown="ignore",
                        sparse_output=False,
                    ),
                    CATEGORICAL_FEATURES,
                ),
            ]
        )
    )


    X_train_structured = (
        preprocessor
        .fit_transform(train_df)
        .astype(np.float32)
    )


    X_validation_structured = (
        preprocessor
        .transform(validation_df)
        .astype(np.float32)
    )


    X_test_structured = (
        preprocessor
        .transform(test_df)
        .astype(np.float32)
    )


    # ========================================================
    # COMBINE WITH TF-IDF
    # ========================================================

    X_train = hstack(
        [
            csr_matrix(
                X_train_structured
            ),

            X_train_tfidf,
        ],
        format="csr",
    )


    X_validation = hstack(
        [
            csr_matrix(
                X_validation_structured
            ),

            X_validation_tfidf,
        ],
        format="csr",
    )


    X_test = hstack(
        [
            csr_matrix(
                X_test_structured
            ),

            X_test_tfidf,
        ],
        format="csr",
    )


    print(
        "Final feature shape:",
        X_train.shape
    )


    # ========================================================
    # LIGHTGBM
    # ========================================================

    model = lgb.LGBMRegressor(

        objective="regression",

        n_estimators=4000,

        learning_rate=0.025,

        num_leaves=63,

        max_depth=-1,

        min_child_samples=20,

        subsample=0.85,

        colsample_bytree=0.85,

        reg_alpha=0.05,

        reg_lambda=1.0,

        random_state=RANDOM_STATE,

        verbosity=-1,

        n_jobs=-1,
    )


    model.fit(

        X_train,

        y_train_log,

        eval_set=[
            (
                X_validation,
                y_validation_log,
            )
        ],

        callbacks=[
            lgb.early_stopping(
                stopping_rounds=150,
                verbose=False,
            )
        ],
    )


    # ========================================================
    # PREDICTIONS
    # ========================================================

    validation_prediction = np.expm1(
        model.predict(
            X_validation
        )
    )


    test_prediction = np.expm1(
        model.predict(
            X_test
        )
    )


    validation_prediction = np.clip(
        validation_prediction,
        0,
        None,
    )


    test_prediction = np.clip(
        test_prediction,
        0,
        None,
    )


    # ========================================================
    # METRICS
    # ========================================================

    validation_metrics = evaluate(
        y_validation,
        validation_prediction,
    )


    test_metrics = evaluate(
        y_test,
        test_prediction,
    )


    print()
    print("VALIDATION")
    print("-" * 50)

    print(
        f"MAE:       "
        f"{validation_metrics['mae']:.4f}"
    )

    print(
        f"RMSE:      "
        f"{validation_metrics['rmse']:.4f}"
    )

    print(
        f"Median AE: "
        f"{validation_metrics['median_ae']:.4f}"
    )

    print(
        f"R²:        "
        f"{validation_metrics['r2']:.4f}"
    )


    print()
    print("TEST")
    print("-" * 50)

    print(
        f"MAE:       "
        f"{test_metrics['mae']:.4f}"
    )

    print(
        f"RMSE:      "
        f"{test_metrics['rmse']:.4f}"
    )

    print(
        f"Median AE: "
        f"{test_metrics['median_ae']:.4f}"
    )

    print(
        f"R²:        "
        f"{test_metrics['r2']:.4f}"
    )


    return {

        "experiment":
            experiment_name,

        "cluster_id":
            use_cluster,

        "validation_mae":
            validation_metrics["mae"],

        "validation_rmse":
            validation_metrics["rmse"],

        "validation_median_ae":
            validation_metrics["median_ae"],

        "validation_r2":
            validation_metrics["r2"],

        "test_mae":
            test_metrics["mae"],

        "test_rmse":
            test_metrics["rmse"],

        "test_median_ae":
            test_metrics["median_ae"],

        "test_r2":
            test_metrics["r2"],
    }


# ============================================================
# RUN ABLATION
# ============================================================

results = []


results.append(

    run_experiment(

        experiment_name=
            "title_tfidf_with_cluster",

        use_cluster=True,
    )
)


results.append(

    run_experiment(

        experiment_name=
            "title_tfidf_without_cluster",

        use_cluster=False,
    )
)


# ============================================================
# RESULTS
# ============================================================

results_df = pd.DataFrame(
    results
)


results_df = (

    results_df

    .sort_values(
        "validation_mae"
    )

    .reset_index(
        drop=True
    )
)


print()
print("=" * 80)
print("CLUSTER ABLATION RESULTS")
print("=" * 80)


display(
    results_df
)


# ============================================================
# CURRENT BENCHMARK
# ============================================================

CURRENT_BEST_MAE = 21.982725


best_result = (
    results_df.iloc[0]
)


improvement = (

    (
        CURRENT_BEST_MAE
        - best_result[
            "validation_mae"
        ]
    )

    / CURRENT_BEST_MAE

    * 100
)


print()
print("=" * 80)
print("DECISION")
print("=" * 80)


print(
    "Current benchmark MAE:",
    CURRENT_BEST_MAE
)


print(
    "Best ablation MAE:",
    round(
        best_result[
            "validation_mae"
        ],
        4,
    )
)


print(
    "Best configuration:",
    best_result[
        "experiment"
    ]
)


print(
    f"Improvement vs benchmark: "
    f"{improvement:.2f}%"
)


if (
    best_result["cluster_id"]
):

    print()
    print(
        "✅ DECISION: KEEP cluster_id"
    )

else:

    print()
    print(
        "✅ DECISION: REMOVE cluster_id"
    )


# ============================================================
# SAVE
# ============================================================

results_df.to_csv(

    REPORT_ROOT
    / "cluster_ablation_results.csv",

    index=False,
)


print()
print(
    "Results saved:",
    REPORT_ROOT
)

DATA
Train: (13984, 26)
Validation: (2997, 26)
Test: (2997, 26)

BUILDING TF-IDF
TF-IDF: (13984, 8000)

EXPERIMENT: title_tfidf_with_cluster
Using cluster_id: True
Numeric features: 27
Final feature shape: (13984, 8267)


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)



VALIDATION
--------------------------------------------------
MAE:       21.8504
RMSE:      85.0096
Median AE: 8.0066
R²:        0.4190

TEST
--------------------------------------------------
MAE:       21.8159
RMSE:      65.9567
Median AE: 7.9940
R²:        0.5158

EXPERIMENT: title_tfidf_without_cluster
Using cluster_id: False
Numeric features: 26
Final feature shape: (13984, 8266)


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)



VALIDATION
--------------------------------------------------
MAE:       27.3113
RMSE:      99.4026
Median AE: 9.3568
R²:        0.2056

TEST
--------------------------------------------------
MAE:       26.4457
RMSE:      76.1600
Median AE: 9.5272
R²:        0.3544

CLUSTER ABLATION RESULTS


,experiment,cluster_id,validation_mae,validation_rmse,validation_median_ae,validation_r2,test_mae,test_rmse,test_median_ae,test_r2
0,title_tfidf_with_cluster,True,21.850382,85.009552,8.006642,0.418996,21.815891,65.956654,7.994022,0.515778
1,title_tfidf_without_cluster,False,27.311327,99.402637,9.356775,0.205600,26.445702,76.160034,9.527222,0.354373



DECISION
Current benchmark MAE: 21.982725
Best ablation MAE: 21.8504
Best configuration: title_tfidf_with_cluster
Improvement vs benchmark: 0.60%

✅ DECISION: KEEP cluster_id

Results saved: /Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/data/reports/price_prediction/cluster_ablation


## what information is actually available in title and product_text

In [11]:
import pandas as pd
from pathlib import Path

CURRENT_DIRECTORY = Path.cwd()

PROJECT_ROOT = (
    CURRENT_DIRECTORY.parent
    if CURRENT_DIRECTORY.name.lower() == "notebooks"
    else CURRENT_DIRECTORY
)

TRAIN_FILE = (
    PROJECT_ROOT
    / "data"
    / "amazon_multimodal"
    / "model_input"
    / "train.parquet"
)

df = pd.read_parquet(TRAIN_FILE)

print("=" * 100)
print("AVAILABLE COLUMNS")
print("=" * 100)

for column in df.columns:
    print(column)


print("\n" + "=" * 100)
print("TITLE vs PRODUCT_TEXT")
print("=" * 100)

for i in range(20):

    print(f"\nPRODUCT {i + 1}")
    print("-" * 100)

    print("TITLE:")
    print(df.iloc[i]["title"])

    print("\nPRODUCT TEXT:")
    print(df.iloc[i]["product_text"])


print("\n" + "=" * 100)
print("TEXT LENGTH STATISTICS")
print("=" * 100)

for column in ["title", "product_text", "model_text"]:

    if column in df.columns:

        lengths = (
            df[column]
            .fillna("")
            .astype(str)
            .str.len()
        )

        print(f"\n{column}")

        print(
            lengths.describe(
                percentiles=[
                    0.50,
                    0.75,
                    0.90,
                    0.95,
                    0.99,
                ]
            )
        )


print("\n" + "=" * 100)
print("EXAMPLE EXPENSIVE PRODUCTS")
print("=" * 100)

display(
    df.sort_values(
        "price",
        ascending=False
    )[
        [
            "asin",
            "title",
            "product_text",
            "category_name",
            "price"
        ]
    ].head(30)
)

AVAILABLE COLUMNS
asin
title
category_name
price
listPrice
stars
reviews
isBestSeller
boughtInLastMonth
product_text
price_band
cluster_id
imgUrl
productURL
absolute_path
download_success
download_status
width
height
file_size_bytes
image_exists
model_text
image_file_exists
original_split_group
split_group
dataset_split

TITLE vs PRODUCT_TEXT

PRODUCT 1
----------------------------------------------------------------------------------------------------
TITLE:
Cute Animal Sloth Earrings，Long Chain Tassel Sloth Drop Earrings for Women Girls Teens Kids Inspired Jewelry

PRODUCT TEXT:
title: Cute Animal Sloth Earrings，Long Chain Tassel Sloth Drop Earrings for Women Girls Teens Kids Inspired Jewelry | category: Boys' Jewelry

PRODUCT 2
----------------------------------------------------------------------------------------------------
TITLE:
Boys Formal Suits Set Outfit with Dress Shirt and Bowtie

PRODUCT TEXT:
title: Boys Formal Suits Set Outfit with Dress Shirt and Bowtie | category: Boy

,asin,title,product_text,category_name,price
13746,B0BSGBKMK8,Synology SA6400 12-Bay Rackmount NAS with Redu...,title: Synology SA6400 12-Bay Rackmount NAS wi...,Data Storage,8499.99
7403,B0B5FL3L6G,Synology 24-Bay FlashStation FS3410 (Diskless),title: Synology 24-Bay FlashStation FS3410 (Di...,Data Storage,6622.60
6786,B083W43YL6,HPE - PROLIANT SERVERS Hewlett Packard Enterpr...,title: HPE - PROLIANT SERVERS Hewlett Packard ...,Computer Servers,2627.77
3337,B00U32IU3O,Hypertherm 088096 Powermax 30 AIR Hand System ...,title: Hypertherm 088096 Powermax 30 AIR Hand ...,Welding & Soldering,2280.00
9553,B09HYQWQ52,"2021 Newest Dell XPS 17 Laptop 9710, 17"" UHD+ ...","title: 2021 Newest Dell XPS 17 Laptop 9710, 17...",Computers & Tablets,2199.00
9141,B0842K1PQ6,Silverstone SST-FAR1W-G - FARA R1 Tower ATX Co...,title: Silverstone SST-FAR1W-G - FARA R1 Tower...,Computer External Components,1803.34
6412,B0BCWNXS6B,Lenovo ThinkPad X13 Yoga Gen 3 21AW002NUS 13.3...,title: Lenovo ThinkPad X13 Yoga Gen 3 21AW002N...,Computers & Tablets,1550.88
6695,B0B791FJHG,"Dell Latitude 7000 7530 15.6"" Notebook - Full ...","title: Dell Latitude 7000 7530 15.6"" Notebook ...",Computers & Tablets,1500.00
10121,B0B3TPYJFZ,HP Z2 G9 Workstation - Intel Core i7 Dodeca-co...,title: HP Z2 G9 Workstation - Intel Core i7 Do...,Computers & Tablets,1432.90
7962,B0BCL3LQ1H,"Lenovo ThinkPad P14s Gen 3 21AK002LUS 14"" Mobi...",title: Lenovo ThinkPad P14s Gen 3 21AK002LUS 1...,Computers & Tablets,1399.00


## check against the original Amazon source dataset

In [12]:
from pathlib import Path
import pandas as pd

CURRENT_DIRECTORY = Path.cwd()

PROJECT_ROOT = (
    CURRENT_DIRECTORY.parent
    if CURRENT_DIRECTORY.name.lower() == "notebooks"
    else CURRENT_DIRECTORY
)

AMAZON_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "amazon"
    / "amazon_price_training.parquet"
)

df = pd.read_parquet(AMAZON_FILE)

print("=" * 80)
print("AMAZON DATASET COLUMNS")
print("=" * 80)

for column in df.columns:
    print(column)

print("\nRows:", f"{len(df):,}")

# Look specifically for useful product attributes
keywords = [
    "brand",
    "manufacturer",
    "description",
    "feature",
    "details",
    "spec",
    "model",
]

print("\n" + "=" * 80)
print("POTENTIALLY USEFUL COLUMNS")
print("=" * 80)

for column in df.columns:
    if any(
        keyword in column.lower()
        for keyword in keywords
    ):
        print(column)

AMAZON DATASET COLUMNS
asin
title
imgUrl
productURL
stars
reviews
price
listPrice
category_id
category_name
isBestSeller
boughtInLastMonth
has_valid_image_url
has_valid_product_url
has_rating
has_reviews
has_valid_price
price_log1p
discount_amount
discount_percentage
title_length
title_word_count
reviews_log1p
bought_log1p

Rows: 1,393,564

POTENTIALLY USEFUL COLUMNS
